In [1]:
import time; T0 = time.time()
!pip install -q deepspeed bitsandbytes peft opensmile
!pip uninstall -y -q torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 19.0 MB/s eta 0:00:00


In [2]:
import os
!git clone -q https://github.com/zehuiwu/SpeechCueLLM /kaggle/working/SpeechCueLLM
os.chdir("/kaggle/working/SpeechCueLLM/LLM_code")

In [3]:
MODEL_PATH = "/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1"   # your working path
EPOCHS = 3
DATASET, LR = "iemocap", "3e-4"        # MELD later: "meld", "2e-4"
assert os.path.exists(MODEL_PATH + "/config.json")

In [4]:
import re, json
s = open("main.py").read()
s = s.replace(
"        model =AutoModelForCausalLM.from_pretrained(args.model_name_or_path).half()",
"""        import time, gc, torch as _t
        if int(os.environ.get("LOCAL_RANK", 0)) == 1:
            time.sleep(180)          # stagger loading to keep CPU RAM low
        from transformers import BitsAndBytesConfig
        _bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=_t.float16)
        model = AutoModelForCausalLM.from_pretrained(args.model_name_or_path, quantization_config=_bnb,
                    torch_dtype=_t.float16, low_cpu_mem_usage=True,
                    device_map={"": int(os.environ.get("LOCAL_RANK", 0))})
        gc.collect()
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=bool(args.gradient_checkpointing))""")
s = s.replace('deepspeed_config["bfloat16"]["enabled"] = True\n        deepspeed_config["fp16"]["enabled"] = False',
              'deepspeed_config["bfloat16"]["enabled"] = False\n        deepspeed_config["fp16"]["enabled"] = True')
s = s.replace("    return optimizer_grouped_parameters",
              "    return [g for g in optimizer_grouped_parameters if len(g['params']) > 0]")
s = s.replace("    deepspeed_config[\"zero_optimization\"]['mics_shard_size'] = world_size", "    pass")
open("main.py", "w").write(s)

p = "data_utils/deepspeed_config.json"
cfg = json.load(open(p)); cfg.pop("amp", None); cfg["zero_optimization"]["stage"] = 2
json.dump(cfg, open(p, "w"), indent=2)
print("4bit", "BitsAndBytesConfig" in s, "| fp16", 'deepspeed_config["fp16"]["enabled"] = True' in s, "| amp removed", "amp" not in cfg)

4bit True | fp16 True | amp removed True


In [5]:
sh = open("train_and_inference.sh").read()
def setv(name, val):
    global sh; sh = re.sub(rf"^{name}=.*$", f"{name}={val}", sh, count=1, flags=re.M)
setv("MODEL_NAME", "'LLaMA2'"); setv("Experiments_setting", "'lora'"); setv("dataset", f"'{DATASET}'")
setv("audio_description", "'True'"); setv("audio_context", "'True'"); setv("audio_impression", "'False'"); setv("audio_only", "'False'")
setv("SEED", 11); setv("num_train_epochs", EPOCHS); setv("LORA_LR", LR)
setv("accumulations", 4); setv("graphics_card", 2); setv("data_percent", 1.0)
sh = sh.replace("MODEL_PATH='LLaMA2 MODELPATH'", f"MODEL_PATH='{MODEL_PATH}'")
sh = sh.replace("--seed ${SEED}\n    fi", "--seed ${SEED} \\\n        --gradient_checkpointing\n    fi")
open("train_and_inference.sh", "w").write(sh)
!grep -n -E "^(MODEL_NAME|Experiments_setting|dataset|audio_|SEED|num_train|LORA_LR|accum|graphics|data_percent)" train_and_inference.sh

10:MODEL_NAME='LLaMA2'
15:Experiments_setting='lora'
21:dataset='iemocap'
25:audio_description='True'
26:audio_impression='False'
27:audio_context='True'
28:audio_only='False'
31:SEED=11
32:num_train_epochs=3
33:LORA_LR=3e-4
40:accumulations=4
41:graphics_card=2
55:data_percent=1.0


In [6]:
import json
p = "data_utils/deepspeed_config.json"
cfg = json.load(open(p))
cfg.pop("amp", None)                         # removed in new DeepSpeed
cfg["zero_optimization"]["stage"] = 2        # keep ZeRO-2 (not 3) with 4-bit
json.dump(cfg, open(p, "w"), indent=2)
print(sorted(cfg.keys()))

['bfloat16', 'data_efficiency', 'data_sampling', 'fp16', 'gradient_accumulation_steps', 'gradient_clipping', 'steps_per_print', 'train_batch_size', 'wall_clock_breakdown', 'zero_allow_untested_optimizer', 'zero_optimization']


In [7]:
import subprocess
LOG = "/kaggle/working/run_log_7B_qlora.txt"
proc = subprocess.Popen(f"PYTHONUNBUFFERED=1 bash train_and_inference.sh > {LOG} 2>&1", shell=True)
print("started, pid", proc.pid)

started, pid 165


In [8]:
DEADLINE_H = 11.0   # kill before Kaggle's 12 h limit so the log and F1 history are kept

def status():
    lines = open(LOG, errors="ignore").read().replace("\r", "\n").split("\n")
    steps = [l for l in lines if "Running Loss" in l]
    last = steps[-1].strip()[-140:] if steps else "(loading model / not started)"
    f1 = [l for l in lines if l.startswith(("current_f1", "best_f1", "history"))][-3:]
    err = [l for l in lines if "Traceback" in l or "OutOfMemory" in l or "Killed" in l][-2:]
    gpu = subprocess.run("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader",
                         shell=True, capture_output=True, text=True).stdout.strip().replace("\n", " | ")
    ram = subprocess.run("free -g | awk 'NR==2{print $3\"/\"$2\" GB\"}'", shell=True, capture_output=True, text=True).stdout.strip()
    print(f"[{time.strftime('%H:%M:%S')} +{(time.time()-T0)/3600:.1f}h] {last}\n  GPU: {gpu} | RAM: {ram}")
    for l in f1 + err: print("  ", l.strip()[:200])

while proc.poll() is None:
    status(); time.sleep(300)
    if time.time() - T0 > DEADLINE_H * 3600:
        print("DEADLINE reached, stopping training"); subprocess.run("pkill -f main.py", shell=True); break
status()
subprocess.run("pkill -f main.py; pkill -f deepspeed", shell=True)
print("FINISHED, exit code", proc.poll())

[19:16:44 +0.0h] (loading model / not started)
  GPU: 0 MiB, 0 % | 0 MiB, 0 % | RAM: 1/31 GB
[19:21:44 +0.1h] (loading model / not started)
  GPU: 6425 MiB, 0 % | 3721 MiB, 30 % | RAM: 3/31 GB
[19:26:44 +0.2h] Epochs 0/3. Running Loss:    1.1173:   6%|▋         | 167/2582 [04:52<1:12:42,  1.81s/it]
  GPU: 5331 MiB, 100 % | 5227 MiB, 100 % | RAM: 5/31 GB
[19:31:44 +0.3h] Epochs 0/3. Running Loss:    1.0346:  13%|█▎        | 328/2582 [09:52<1:06:58,  1.78s/it]
  GPU: 5417 MiB, 100 % | 5227 MiB, 100 % | RAM: 5/31 GB
[19:36:44 +0.3h] Epochs 0/3. Running Loss:    0.9635:  19%|█▉        | 490/2582 [14:52<1:10:45,  2.03s/it]
  GPU: 5417 MiB, 100 % | 5227 MiB, 100 % | RAM: 5/31 GB
[19:41:44 +0.4h] Epochs 0/3. Running Loss:    1.3191:  25%|██▌       | 651/2582 [19:53<1:01:34,  1.91s/it]
  GPU: 5417 MiB, 94 % | 5227 MiB, 99 % | RAM: 5/31 GB
[19:46:44 +0.5h] Epochs 0/3. Running Loss:    0.6113:  31%|███▏      | 811/2582 [24:52<53:47,  1.82s/it]
  GPU: 5417 MiB, 100 % | 5227 MiB, 100 % | RAM: 5/31